In [68]:
import os
import json

with open("../env/keys.json", "r", encoding="utf-8") as f:
    keys = json.load(f)

MISTRAL_KEY = keys["MISTRAL_KEY"]
HF_KEY = keys["HF_TOKEN"]

os.environ["MISTRAL_API_KEY"] = MISTRAL_KEY

In [69]:
import time

In [70]:
from mistralai import Mistral


#api_key = os.environ.get("MISTRAL_API_KEY")
api_key = MISTRAL_KEY
client = Mistral(api_key=api_key)
model = "mistral-large-latest"

client = Mistral(api_key=api_key)


with open("incidents_categories.json", "r", encoding="utf-8") as f:
    incident_data = json.load(f)

with open("incidents_arbo_simple.json", "r", encoding="utf-8") as f:
    incident_arbo = json.load(f)

Localisation

In [19]:
prompt = f""" Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en la localisation de l'incident.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{incident_data["Localisation (QR)"]}

Réponds exactement par la localisation comme écrite dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse.
Voici la transcription audio :
"La lunette des toilettes glisse dans le wagon 3."
"""

WC


Catégorie

In [21]:
prompt = f""" Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraite la catégorie de l'incident.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{incident_data["Catégorie"]}

Réponds exactement par la catégorie comme écrite dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse.
Voici la transcription audio :
"La lunette des toilettes glisse dans le wagon 3."
"""

Localisation N+2

In [23]:
prompt = f""" Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraite la localisation exacte de l'incident.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{incident_data["Précision à localisation N2"]}

Réponds exactement par la localisation comme écrite dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse.
Voici la transcription audio :
"La lunette des toilettes glisse dans le wagon 3."
"""

Organe

In [26]:
prompt = f""" Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraite l'objet exact concerné par l'incident.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{set([keys for keys in incident_arbo["Sanitaire"]["WC"].keys()])}

Réponds exactement par l'objet comme écrit dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse.
Voici la transcription audio :
"La lunette des toilettes glisse dans le wagon 3."
"""

In [59]:
def call_mistral(prompt):
    chat_response = client.chat.complete(
    model=model,
    messages=[
        {"role": "user", "content": prompt},
    ]
)
    return chat_response.choices[0].message.content

print(call_mistral(prompt))

Je ne sais pas.


In [73]:
message1 = "La lunette des toilettes glisse dans le wagon 3."

message2 = "Alors dans la salle, l'accessoire casier des bagages au niveau de l'escalier à gauche du wagon 6, il y a un tag"

message3 = "La tablette du siège 53 du wagon 2 est cassée."

message = message2

Tous ensemble 

In [72]:
def get_all_keys_recursive(d):
    """Récupère tous les niveaux de l'arborescence sous forme de listes uniques."""
    keys_level_1 = list(d.keys())
    keys_level_2 = list({k for v in d.values() for k in v.keys()})
    keys_level_3 = list({k for v in d.values() for sub in v.values() for k in sub.keys()})
    #keys_level_4 = list({item for v in d.values() for sub in v.values() for val in sub.values() for item in val})
    return keys_level_1, keys_level_2, keys_level_3

# Récupérer toutes les options possibles
all_localisations, all_categories, all_objets = get_all_keys_recursive(incident_arbo)

# Étape 1 : Localisation
prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire la localisation de l'incident.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{incident_data["Localisation (QR)"]}

Réponds exactement par la lsite de localisations comme écrite dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les localisations par ";".
Voici la transcription audio :
{message}
"""
localisation = call_mistral(prompt)
print(localisation)
time.sleep(5)

if "sais pas" in localisation:
    localisation = all_localisations

# Étape 2 : Catégorie
categories_possibles = set()
localisations = localisation.split(";")
for loc in localisations:
    loc = loc.strip()
    if loc in incident_arbo:
        categories_possibles.update(incident_arbo[loc].keys())
    else:
        print(f"Localisation '{loc}' non trouvée dans l'arborescence.")
        categories_possibles = set(incident_arbo.get(loc, {}).keys())
# if isinstance(localisation, list):
#     for loc in localisation:
#         categories_possibles.update(incident_arbo.get(loc, {}).keys())
# else:
#     categories_possibles = set(incident_arbo.get(localisation, {}).keys())

print(categories_possibles)
prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire la catégorie exacte concernée par l'incident.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{categories_possibles}

Réponds exactement par la liste de catégories comme écrite dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les catégories par ";".
Voici la transcription audio :
{message}
"""
categorie = call_mistral(prompt)
print(categorie)
time.sleep(5)

if "sais pas" in categorie:
    categorie = all_categories

categories = categorie.split(";")

# Étape 3 : Objet
objets_possibles = set()
for cat in categories:
    cat = cat.strip()
    if cat in incident_arbo.get(localisation, {}):
        objets_possibles.update(incident_arbo[localisation].get(cat, {}).keys())
    else:
        print(f"Catégorie '{cat}' non trouvée dans l'arborescence.")
        objets_possibles = set(incident_arbo.get(localisation, {}).get(cat, {}).keys())
# if isinstance(localisation, list) or isinstance(categorie, list):
#     for loc in localisation if isinstance(localisation, list) else [localisation]:
#         for cat in categorie if isinstance(categorie, list) else [categorie]:
#             objets_possibles.update(incident_arbo.get(loc, {}).get(cat, {}).keys())
# else:
#     objets_possibles = set(incident_arbo.get(localisation, {}).get(categorie, {}).keys())

print(objets_possibles)
prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire l'objet exact concerné par l'incident.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{objets_possibles}

Réponds exactement par la liste d'objets comme écrit dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les objets par ";".
Voici la transcription audio :
{message}
"""
objet = call_mistral(prompt)
print(objet)
time.sleep(5)

if "sais pas" in objet:
    objet = all_objets

# Étape 4 : Nature du problème
problemes_possibles = set()
if isinstance(localisation, list) or isinstance(categorie, list) or isinstance(objet, list):
    for loc in localisation if isinstance(localisation, list) else [localisation]:
        for cat in categorie if isinstance(categorie, list) else [categorie]:
            for obj in objet if isinstance(objet, list) else [objet]:
                problemes_possibles.update(incident_arbo.get(loc, {}).get(cat, {}).get(obj, []))
else:
    problemes_possibles = incident_arbo.get(localisation, {}).get(categorie, {}).get(objet, [])

print(problemes_possibles)
prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire la nature de l'incident.
Celui-ci concerne la localisation {localisation}, la catégorie {categorie} et l'objet {objet}.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{problemes_possibles}

Réponds exactement par la liste de problèmes comme écrit dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les problèmes par ";".
Voici la transcription audio :
{message}
"""
probleme = call_mistral(prompt)
print(probleme)


Sanitaire
{'Pack inoui', 'Porte local', 'Eclairage', 'Accessoires/ Environnement', 'Lave-mains', 'Habillage', 'Climatisation', 'Prise 220 Volts', 'Info/Communication', 'WC'}
WC
{'Réservoir de rétention', "Chasse d'eau", 'Odeurs', 'Ensemble cuvette', 'Système WC supérieur', 'Système WC inférieur'}
Je ne sais pas
{',', 'M', 'A', '-', 'd', 'C', 'R', 'é', 'v', 'q', 'h', 'g', 'W', '0', 'f', 'O', 'u', 'S', ' ', 'I', 'N', 'r', 'x', 'n', 'm', 't', '1', 'i', 'e', 'a', 'o', 's', 'c'}
Je ne sais pas


In [ ]:
def get_all_keys_recursive(d):
    """Récupère tous les niveaux de l'arborescence sous forme de listes uniques."""
    keys_level_1 = list(d.keys())
    keys_level_2 = list({k for v in d.values() for k in v.keys()})
    keys_level_3 = list({k for v in d.values() for sub in v.values() for k in sub.keys()})
    return keys_level_1, keys_level_2, keys_level_3


def parse_response(response, default_list, prob):
    """Nettoie et découpe une réponse du modèle."""
    if "sais pas" in response.lower():
        return default_list
    if prob:
        return response
    if "liste" in response.lower():
        response = response.split("\n")[1:]
    return normalize_to_list(response)


def call_step(prompt, default_list, prob=False):
    """Appelle Mistral et gère la réponse."""
    response = call_mistral(prompt)
    print(response)
    time.sleep(5)
    if not prob:
        return parse_response(response, default_list, prob)
    else:
        return response

def normalize_to_list(value):
    """Transforme une chaîne séparée par ; en liste, ou retourne la liste telle quelle."""
    if isinstance(value, str):
        return [v.strip() for v in value.split(";")]
    elif isinstance(value, list):
        return value
    else:
        return []

# Récupérer toutes les options possibles
all_localisations, all_categories, all_objets = get_all_keys_recursive(incident_arbo)

# Étape 1 : Localisation
prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire la localisation de l'incident.
Tu dois répondre par une liste de localisations possibles, en te basant sur les données d'entraînement fournies.
Voici les localisations possibles :
{incident_data["Localisation (QR)"]}

Réponds exactement par la liste de localisations comme écrite dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les localisations par ";".
Voici la transcription audio :
{message}
"""
localisations = call_step(prompt, all_localisations)

# Étape 2 : Catégorie
categories_possibles = set()
for loc in localisations:
    categories_possibles.update(incident_arbo.get(loc, {}).keys())

prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire la catégorie exacte concernée par l'incident.
Tu dois répondre par une liste de catégories possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories possibles :
{categories_possibles}

Réponds exactement par la liste de catégories comme écrite dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les catégories par ";".
Voici la transcription audio :
{message}
"""
categories = call_step(prompt, all_categories)

# Étape 3 : Objet
objets_possibles = set()
for loc in localisations:
    for cat in categories:
        objets_possibles.update(incident_arbo.get(loc, {}).get(cat, {}).keys())

prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire l'objet exact concerné par l'incident.
Tu dois répondre par une liste d'objets possibles, en te basant sur les données d'entraînement fournies.
Voici les objets possibles :
{objets_possibles}

Réponds exactement par la liste d'objets comme écrit dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les objets par ";".
Voici la transcription audio :
{message}
"""
objets = call_step(prompt, all_objets)

# Étape 4 : Nature du problème
problemes_possibles = set()
for loc in localisations:
    for cat in categories:
        for obj in objets:
            problemes_possibles.update(incident_arbo.get(loc, {}).get(cat, {}).get(obj, []))
print(problemes_possibles)
if len(problemes_possibles) == 1:
    problemes = problemes_possibles
else:
    prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire la nature de l'incident.
    Celui-ci concerne les localisations {localisations}, les catégories {categories} et les objets {objets}.
    Tu dois répondre par une liste de natures d'incidents possibles, en te basant sur les données d'entraînement fournies.
    Voici les natures d'incident possibles :
    {problemes_possibles}

    Réponds exactement par la liste de problèmes comme écrit dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les problèmes par ";".
    Voici la transcription audio :
    {message}
    """
    problemes = call_step(prompt, list(problemes_possibles), prob=True)

# Résumé final
print("\nRésultat final :")
print("Localisations :", localisations)
print("Catégories     :", categories)
print("Objets         :", objets)
print("Problèmes      :", problemes)


Voici la liste de localisations possibles :

Salle


AttributeError: 'list' object has no attribute 'split'

Appel Mistral

Ensemble cuvette / Lunette/Couvercle WC
